# Sound Processing Lab - Part #5
**⚠️ DISCLAIMER: THIS IS INTENDED FOR LEARNING PURPOSES ONLY, AND SHOULD NEVER BE USED WITH THE INTENTION OF FELONY OR TO INFLICT DAMAGE TO OTHERS.**
## Dataset & Ethics — Read This Before Continuing
We use two datasets, both released under permissive licenses for research and education:

- **LibriSpeech** (CC BY 4.0) — read English speech from public-domain audiobooks

We deliberately **do not** use voices scraped from YouTube, social media, podcasts, or any source where the speaker did not knowingly consent to have their voice used to train models. The human voice is biometric data. In Argentina it is protected under **Argentine Law 25.326** (Personal Data Protection); in the EU under **GDPR Article 9** (special-category data). Treat it accordingly throughout your career.


**In this notebook you'll learn:**  
How to build a **voice biometrics system** — a database of speaker identities that can authenticate a person from a short voice sample, the same way Apple Voice ID or banking voice-auth systems work. By the end of this notebook you will be able to enroll a speaker, persist their voice fingerprint to a database, and verify a new voice sample against the enrolled identities.

### Learning objectives
1. Load, inspect, and clean raw audio signals in Python
2. Extract **MFCC** features and visualise what the model sees internally
3. Generate **speaker embeddings** with a pretrained deep encoder (WavLM)
4. Design a **SQLite biometrics database** to persist enrolled voice fingerprints
5. Calibrate a verification threshold using **EER** and **FAR=1%** — the standard metrics in production biometrics
6. Detect **deepfake audio** and apply **watermarking** techniques (classical + neural with AudioSeal)

## ⚙️ Environment Setup

In [ ]:
# Run once per session. On Google Colab the install takes ~60s; locally, faster if cached.
import os
from dotenv import load_dotenv, find_dotenv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
import torch
from scipy import signal as scipy_signal
from pathlib import Path
from transformers import AutoFeatureExtractor, WavLMForXVector
import pyaudio
import time
from dataclasses import dataclass
from typing import Optional
from IPython.display import Audio, display
from audioseal import AudioSeal
import torch._dynamo
from scipy.signal import iirnotch, filtfilt
from itertools import combinations
import sqlite3
from datetime import datetime
import re
import torchaudio  # only used for the dataset *downloader*, not for I/O
print("Downloading LibriSpeech test-clean (first run only)...")

# Reproducibility — fixed seed so plots and numerical outputs match across everybody's machines
SEED = 42
np.random.seed(SEED)

load_dotenv(find_dotenv())   # load GOOGLE_API_KEY from .env
if not os.environ.get("GOOGLE_API_KEY"):
    print("⚠️  GOOGLE_API_KEY not set in script. Set it before running cells that call Gemini.\n")

# Plot defaults — readable on a projector
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["figure.dpi"]     = 110
plt.rcParams["axes.grid"]      = True
plt.rcParams["grid.alpha"]     = 0.3

# Working directories — keep all artefacts under one folder so cleanup is trivial
WORKSHOP_DIR   = Path("..").resolve()
AUDIO_DIR      = WORKSHOP_DIR / "audio"        # raw .wav files
FEATURES_DIR   = WORKSHOP_DIR / "features"     # MFCC matrices (.npy)
EMBEDDINGS_DIR = WORKSHOP_DIR / "embeddings"   # WavLM speaker embeddings (.npy)
MODELS_DIR     = WORKSHOP_DIR / "models"       # pretrained model weights (AudioSeal, etc.)
DB_PATH        = WORKSHOP_DIR / "biometrics.db"  # SQLite biometrics database

for d in (AUDIO_DIR, FEATURES_DIR, EMBEDDINGS_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"librosa  : {librosa.__version__}")
print(f"numpy    : {np.__version__}")
print(f"torch    : {torch.__version__}")
print(f"work dir : {WORKSHOP_DIR}")
print(f"db path  : {DB_PATH}")

## 1. Acquiring the Dataset

LibriSpeech is split into several subsets. We use **`test-clean`** (~346 MB, ~5.4 hours of speech from 40 speakers) because:

- It is the smallest "clean" (low-noise) subset — fast enough to download once and forget
- The directory structure groups files per-speaker, which makes biometric enrollment experiments trivial to set up
- It is the de-facto benchmark — any paper you read on speaker recognition reports numbers on it

From the 5.4-hour corpus we hand-pick **10 speakers × 3 utterances = 30 clips** — enough to populate a biometrics database with statistical mass to calibrate a real verification threshold (we'll get **30 genuine pairs** and **405 impostor pairs** out of this, which is industry-grade on a toy scale).

After we extract the 30 clips we care about, we delete the rest of the corpus. This isn't just disk hygiene — it models the real-world workflow of voice enrollment: **ingest raw audio, extract the biometric template, persist it, discard the raw audio**. In production biometrics, retaining raw voice samples is a liability (GDPR Art. 9, Argentina's Ley 25.326 — voice is sensitive biometric data).

> **Speaker IDs in LibriSpeech are anonymous integers** (e.g. `1089`, `1188`). The speakers are real people — volunteer readers from LibriVox — but they consented to public release of their recordings under CC BY 4.0. This is the consent chain we talked about in the intro.

In [ ]:
# Download LibriSpeech test-clean and curate our working sample set
# NOTE: torchaudio 2.x no longer ships a built-in audio decoder. We use
# soundfile (libsndfile) for I/O — it handles FLAC and WAV natively with
# zero system dependencies.

# 1) Download (cached after first run). ~346 MB.
LIBRISPEECH_ROOT = WORKSHOP_DIR / "librispeech"
LIBRISPEECH_ROOT.mkdir(exist_ok=True)

_ = torchaudio.datasets.LIBRISPEECH(
    root=str(LIBRISPEECH_ROOT),
    url="test-clean",
    download=True,
)
print("Done.\n")

# 2) Discover speakers
ls_root = LIBRISPEECH_ROOT / "LibriSpeech" / "test-clean"
all_speakers = sorted(p.name for p in ls_root.iterdir() if p.is_dir())
print(f"Found {len(all_speakers)} speakers in test-clean.\n")

# 3) Pick 10 speakers deterministically (alphabetical = reproducible across student machines)
N_SPEAKERS_TO_USE        = 10
N_UTTERANCES_PER_SPEAKER = 3
SELECTED_SPEAKERS        = all_speakers[:N_SPEAKERS_TO_USE]

print(f"Selected speakers      : {SELECTED_SPEAKERS}")
print(f"Utterances per speaker : {N_UTTERANCES_PER_SPEAKER}")
print(f"Total clips            : {N_SPEAKERS_TO_USE * N_UTTERANCES_PER_SPEAKER}\n")

# 4) Convert .flac -> .wav using soundfile and log everything we ingest
ingestion_log = []
for spk in SELECTED_SPEAKERS:
    spk_dir = ls_root / spk
    flac_files = sorted(spk_dir.rglob("*.flac"))[:N_UTTERANCES_PER_SPEAKER]

    for i, src in enumerate(flac_files, start=1):
        audio, sr = sf.read(src)                  # libsndfile decodes FLAC directly
        dst = AUDIO_DIR / f"speaker_{spk}_utt_{i}.wav"
        sf.write(dst, audio, sr, subtype="PCM_16")
        ingestion_log.append({
            "speaker_id":  spk,
            "utterance":   i,
            "filename":    dst.name,
            "duration_s":  round(len(audio) / sr, 2),
            "sample_rate": sr,
            "size_kb":     round(dst.stat().st_size / 1024, 1),
        })

# 5) Show the ingestion as a clean table
df_ingestion = pd.DataFrame(ingestion_log)
print(f"Ingested {len(df_ingestion)} audio clips.\n")
df_ingestion

## 2. From raw audio to MFCC features

You already know MFCC extraction from Chapters 2-3. We compress the whole pipeline — *load → normalise → trim silence → MFCC → save* — into one function and run it across all **30 utterances**. The output is 30 `.npy` files in `features/`, each containing a `(n_mfcc, n_frames)` matrix.

**Where MFCCs fit in this notebook.** MFCCs are *not* what we'll persist in our biometrics database — modern systems store deep-learning embeddings (next section), not raw spectral features. We extract MFCCs here for two reasons:

1. **Inspection.** They are human-interpretable: you can look at an MFCC matrix and see speech structure. Embeddings are 512 opaque numbers.
2. **Baseline for comparison.** Later in this chapter we'll briefly compare how well naive MFCC-based matching works vs. WavLM embeddings on the same task. The gap is dramatic — and it's the whole reason the field moved to deep learning.

**Why we save to disk.** Feature extraction is deterministic and slow. Embedding extraction (next section) is what we'll iterate on. Caching features keeps the experiment loop tight.

**Parameter choices** (sane defaults for 16 kHz speech, no need to tune today):

| Parameter      | Value | Why                                                   |
|----------------|-------|-------------------------------------------------------|
| `n_mfcc`       | 40    | Standard for speaker recognition (vs. 13 for ASR)     |
| `n_fft`        | 512   | 32 ms window at 16 kHz — captures one pitch period    |
| `hop_length`   | 160   | 10 ms stride — standard frame rate                    |
| `top_db` (trim)| 30    | Removes leading/trailing silence below 30 dB of peak  |

In [ ]:
# Audio -> MFCC pipeline, applied to all 30 utterances
SR_TARGET   = 16_000
N_MFCC      = 40
N_FFT       = 512
HOP_LENGTH  = 160
TRIM_TOP_DB = 30

def audio_to_mfcc(wav_path: Path) -> np.ndarray:
    """Load a wav, resample to 16 kHz mono, trim silence, return MFCC matrix."""
    y, sr = librosa.load(wav_path, sr=SR_TARGET, mono=True)
    y, _  = librosa.effects.trim(y, top_db=TRIM_TOP_DB)
    mfcc  = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH
    )
    return mfcc  # shape: (N_MFCC, n_frames)

# Run pipeline on every wav, save features, collect metadata
feature_log = []
for wav_path in sorted(AUDIO_DIR.glob("speaker_*.wav")):
    parts   = wav_path.stem.split("_")
    spk_id  = parts[1]
    utt_idx = int(parts[3])

    mfcc      = audio_to_mfcc(wav_path)
    feat_path = FEATURES_DIR / f"{wav_path.stem}.npy"
    np.save(feat_path, mfcc)

    feature_log.append({
        "speaker_id": spk_id,
        "utterance":  utt_idx,
        "filename":   wav_path.name,
        "mfcc_shape": str(mfcc.shape),
        "n_frames":   mfcc.shape[1],
        "feat_kb":    round(feat_path.stat().st_size / 1024, 1),
    })

df_features = pd.DataFrame(feature_log)
print(f"Extracted MFCC features for {len(df_features)} utterances, saved to {FEATURES_DIR}\n")
df_features

### Visualising MFCCs across speakers

Let's see the MFCC fingerprints side by side — one utterance per speaker. If MFCCs capture *any* speaker-discriminative information, the textures should look visibly different from row to row.

Spoiler: they do, but the differences are subtle. That's exactly why a deep learning encoder beats raw MFCC for biometric matching — it amplifies the signal that's hard to see by eye.

In [ ]:
# MFCC heatmaps — one per speaker, first utterance
fig, axes = plt.subplots(5, 2, figsize=(14, 12), sharex=True, sharey=True)

for ax, spk in zip(axes.flat, SELECTED_SPEAKERS):
    mfcc = np.load(FEATURES_DIR / f"speaker_{spk}_utt_1.npy")
    img = librosa.display.specshow(
        mfcc, x_axis="time", sr=SR_TARGET, hop_length=HOP_LENGTH, ax=ax,
    )
    ax.set_title(f"Speaker {spk} — utt 1   |   shape={mfcc.shape}")
    ax.set_ylabel("MFCC coeff")

fig.colorbar(img, ax=axes, format="%+0.1f", shrink=0.8)
plt.suptitle("MFCC fingerprints across all 10 enrolled speakers", y=1.00, fontsize=14)
plt.show()

## 3. From MFCCs to speaker embeddings — closing the gap

Open any of your saved MFCC files: shape is `(40, n_frames)` where `n_frames` depends on how long the utterance was. That's the problem we solve in this section.

### The verification question requires a *fixed-length* representation

You cannot answer *"are speaker A and speaker B the same person?"* by comparing two matrices of different shapes. You need a function:

```
f(audio) -> v ∈ ℝᵈ        (same d for every input, regardless of duration)
```

A naive attempt — averaging MFCCs across the time axis to get a 40-dim vector — throws away almost all the discriminative information. It works at roughly 70% accuracy on VoxCeleb. State-of-the-art is above 99%. The gap is enormous and entirely about *how you collapse the time axis*.

### The modern answer: transformer-based speaker encoders

Until ~2020 the state-of-the-art was **ECAPA-TDNN** (Desplanques et al.) — a time-delay convolutional network trained with discriminative loss on VoxCeleb. It produces 192-dim embeddings and is still an excellent baseline.

Since 2021 the field has moved to **self-supervised transformer encoders** fine-tuned for speaker verification. We use **WavLM-Base-Plus-SV** (Chen et al., Microsoft 2022):

1. **Self-supervised pretraining** on 94k hours of unlabelled audio (LibriLight + GigaSpeech + VoxPopuli) teaches the model rich, general-purpose audio representations *before* it ever sees a speaker label.
2. **Fine-tuning** on VoxCeleb1 with an **X-Vector head** + Additive Margin Softmax loss specialises the embedding space for speaker discrimination.
3. **Output**: a 512-dim vector per utterance. Cosine similarity in this space approximates speaker identity.

WavLM beats ECAPA on the standard VoxCeleb1-O benchmark (0.65% EER vs 0.69% EER). More importantly for this workshop: it loads through the mainstream `transformers` library with zero platform-specific hacks.

We use it as a frozen feature extractor — no training. Loading the pretrained weights and running inference takes ~30 seconds on a laptop CPU.

### What we do here

Download the pretrained **WavLM-Base-Plus-SV** model from HuggingFace (~377 MB, Microsoft, MIT license), run it on each of our **30 utterances**, save the 512-dim embeddings to `embeddings/`. These embeddings are the **biometric templates** we'll persist to the SQLite database in the next section — they are what a real production voice biometrics system would store per enrolled user.

No training, no fine-tuning. Training a speaker encoder from scratch would take ~3 days on 4 GPUs; using a pretrained one takes 30 seconds on a laptop.

In [ ]:
# Speaker embeddings via WavLM-Base-Plus-SV
MODEL_ID = "microsoft/wavlm-base-plus-sv"

# 1) Load the feature extractor + model. First call downloads ~377 MB.
print(f"Loading {MODEL_ID} (first run downloads ~377 MB)...")
device = "cuda" if torch.cuda.is_available() else "cpu"

feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)
encoder           = WavLMForXVector.from_pretrained(MODEL_ID).to(device).eval()
print(f"Model loaded on device: {device}")
print(f"Embedding dim: {encoder.config.xvector_output_dim}\n")


def wav_to_embedding(wav_path: Path) -> np.ndarray:
    """Load a wav, resample if needed, return an L2-normalised x-vector."""
    audio, sr = sf.read(wav_path)
    if sr != SR_TARGET:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=SR_TARGET)
    # Mono float32 in [-1, 1] is what the feature extractor expects.
    inputs = feature_extractor(
        audio, sampling_rate=SR_TARGET, return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        emb = encoder(**inputs).embeddings        # shape (1, 512)
    emb = emb.squeeze().cpu().numpy()             # shape (512,)
    emb = emb / np.linalg.norm(emb)               # L2-normalise
    return emb


# Extract and save embeddings for all 30 LibriSpeech utterances
embedding_log = []
for wav_path in sorted(AUDIO_DIR.glob("speaker_[0-9]*.wav")):
    parts   = wav_path.stem.split("_")
    spk_id  = parts[1]
    utt_idx = int(parts[3])

    emb      = wav_to_embedding(wav_path)
    out_path = EMBEDDINGS_DIR / f"{wav_path.stem}.npy"
    np.save(out_path, emb)

    embedding_log.append({
        "speaker_id": spk_id,
        "utterance":  utt_idx,
        "filename":   wav_path.name,
        "dim":        emb.shape[0],
        "norm":       round(float(np.linalg.norm(emb)), 4),
        "emb_bytes":  out_path.stat().st_size,
    })

df_embeddings = pd.DataFrame(embedding_log)
print(f"\nSaved {len(df_embeddings)} embeddings to {EMBEDDINGS_DIR}")
print(f"Each embedding: 512-dim float32 = {512*4} bytes\n")
df_embeddings

## 4. Does the embedding space actually cluster by speaker?

We have **30 L2-normalised 512-dim vectors** (10 speakers × 3 utterances). If WavLM's embedding space is doing what we claim — encoding speaker identity in geometry — then:

- Vectors from the **same speaker** should be close together (cosine similarity near 1)
- Vectors from **different speakers** should be far apart (cosine similarity near 0, or even negative)

Because every embedding is L2-normalised, cosine similarity reduces to a plain dot product:

$$
\cos(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|} = \mathbf{u} \cdot \mathbf{v}
$$

We compute the full **30×30 similarity matrix** and plot it as a heatmap. The visual prediction: **ten bright 3×3 blocks on the diagonal** (intra-speaker pairs), darker everywhere else (inter-speaker pairs). If that pattern is missing, something is wrong with the pipeline.

This is also the last sanity check before we commit these vectors to the SQLite biometrics database in the next section. Garbage embeddings in, garbage biometrics out.

In [ ]:
# Compute and visualise the speaker-embedding similarity matrix

# 1) Load all LibriSpeech embeddings into a single matrix E of shape (30, 512)
embedding_files = sorted(EMBEDDINGS_DIR.glob("speaker_[0-9]*.npy"))
labels    = [f.stem for f in embedding_files]
speakers  = [lbl.split("_")[1] for lbl in labels]
E         = np.stack([np.load(f) for f in embedding_files])
print(f"Embedding matrix E shape: {E.shape}")

# 2) Cosine similarity matrix.
#    Since rows of E are L2-normalised, E @ E.T gives cosine similarities directly.
S = E @ E.T
print(f"Similarity matrix S shape: {S.shape}")
print(f"Diagonal (self-similarity, should all be 1.0): unique values = {np.unique(np.diag(S).round(4))}\n")

# 3) Plot — no per-cell text annotations (too many for 30x30 to be legible)
fig, ax = plt.subplots(figsize=(12, 11))
im = ax.imshow(S, cmap="viridis", vmin=-0.2, vmax=1.0)

short = [f"{s}·u{lbl.split('_')[3]}" for s, lbl in zip(speakers, labels)]
ax.set_xticks(range(len(labels))); ax.set_xticklabels(short, rotation=90, ha="center", fontsize=8)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(short, fontsize=8)

# Block separators — computed from data, robust to N_UTTERANCES changes
unique_speakers = []
for s in speakers:
    if s not in unique_speakers:
        unique_speakers.append(s)
boundaries = np.cumsum([speakers.count(s) for s in unique_speakers])[:-1]
for k in boundaries:
    ax.axhline(k - 0.5, color="red", lw=1.2)
    ax.axvline(k - 0.5, color="red", lw=1.2)

ax.set_title(f"Cosine similarity between WavLM speaker embeddings\n"
             f"({len(unique_speakers)} speakers × {speakers.count(unique_speakers[0])} utterances — red lines separate speakers)")
fig.colorbar(im, ax=ax, label="cosine similarity")
plt.tight_layout()
plt.show()

# 4) Quantitative summary — same-speaker vs different-speaker similarity distributions
same_spk_mask = np.array([[speakers[i] == speakers[j] for j in range(len(labels))]
                          for i in range(len(labels))])
off_diag  = ~np.eye(len(labels), dtype=bool)
same_vals = S[same_spk_mask & off_diag]
diff_vals = S[~same_spk_mask & off_diag]

print(f"Same-speaker pairs      (n={len(same_vals):>3}): mean={same_vals.mean():.3f}  min={same_vals.min():.3f}  max={same_vals.max():.3f}")
print(f"Different-speaker pairs (n={len(diff_vals):>3}): mean={diff_vals.mean():.3f}  min={diff_vals.min():.3f}  max={diff_vals.max():.3f}")
print(f"\nGap between distributions: {same_vals.min() - diff_vals.max():+.3f}")
print("(positive gap = perfect separation; negative = at least one impostor pair scores higher than the worst genuine pair)")

## Part 2 — Enrolling yourself in the biometric database

You've seen what speaker embeddings look like for ten strangers from LibriSpeech. Now you become the eleventh identity in the database — but this time with a **real name** instead of an anonymous integer ID.

You'll record three short clips (~10 seconds each) reading the same sentence. Same pipeline as before — load, extract WavLM embedding, save to disk. This mirrors the standard commercial voice biometrics enrollment flow ("say the passphrase three times"): three samples give the system enough signal to average out per-utterance noise and produce a robust speaker template.

### Consent (read this before recording)

You're about to record your own biometric data and save it to disk on this laptop. You are doing so voluntarily, knowing that:

- The recordings stay **local** — nothing leaves your machine.
- Your three audio clips and the resulting WavLM embedding will be persisted into a SQLite database (`biometrics.db`) under a name you choose yourself.
- At the end of the workshop you should delete `audio/`, `embeddings/`, and `biometrics.db` if you don't want the data persisted.

If any of that is not OK with you, **don't run the next cell**. Skip the enrollment and work with the 10 LibriSpeech speakers only. That's a valid choice and the verification demos still work.

In [ ]:
# Record your own voice (enrollment)
# Records 3 clips of ~10 seconds each. Between clips, you get a 3-second pause
# to breathe. Each clip should be you reading the SAME sentence — repeating the
# same words gives the embedding model the cleanest signal to work with.

# 1) Ask the student for their name
print("=" * 70)
print("BIOMETRIC ENROLLMENT")
print("=" * 70)
STUDENT_NAME = input("\nEnter your name (this is how you'll appear in the DB): ").strip()
if not STUDENT_NAME:
    raise ValueError("Name cannot be empty.")

# Sanitise for use in filenames: lowercase, alphanumeric + underscores only
STUDENT_ID = re.sub(r"[^a-z0-9]+", "_", STUDENT_NAME.lower()).strip("_")
if not STUDENT_ID:
    raise ValueError(f"Name '{STUDENT_NAME}' has no usable characters for a filename.")

print(f"\n  Display name (for DB) : {STUDENT_NAME}")
print(f"  File-safe ID          : {STUDENT_ID}")

# 2) Recording parameters
REFERENCE_SENTENCE = (
    "The quick brown fox jumps over the lazy dog while the rain in Spain "
    "falls mainly on the plain. I am recording my voice for a workshop on "
    "voice processing and deepfake detection."
)

SECONDS_PER_CLIP = 10
N_CLIPS          = 3
MIC_RATE         = 16_000     # WavLM expects 16 kHz
CHUNK_SIZE       = 1024


def record_clip(duration_sec: int, sample_rate: int) -> np.ndarray:
    """Record from the default microphone, return a float32 numpy array in [-1, 1]."""
    pya = pyaudio.PyAudio()
    stream = pya.open(
        format=pyaudio.paInt16, channels=1, rate=sample_rate,
        input=True, frames_per_buffer=CHUNK_SIZE,
    )
    frames = []
    n_chunks = int(sample_rate * duration_sec / CHUNK_SIZE)
    for _ in range(n_chunks):
        frames.append(stream.read(CHUNK_SIZE, exception_on_overflow=False))
    stream.stop_stream(); stream.close(); pya.terminate()

    raw = b"".join(frames)
    audio_int16 = np.frombuffer(raw, dtype=np.int16)
    audio = audio_int16.astype(np.float32) / 32768.0
    return audio


# 3) Instructions
print("\n" + "=" * 70)
print(f"You will record {N_CLIPS} clips of {SECONDS_PER_CLIP} seconds each.")
print(f"For every clip, read this sentence at a normal pace:\n")
print(f'  "{REFERENCE_SENTENCE}"\n')
print("Speak naturally. Don't whisper, don't shout. If you stumble, keep going —")
print("a small mistake won't hurt the embedding.")
input("\nPress ENTER when you're ready to start recording clip 1... ")

# 4) Record
recording_log = []
for i in range(1, N_CLIPS + 1):
    print(f"\n>>> Recording clip {i}/{N_CLIPS} — speak NOW...")
    audio = record_clip(SECONDS_PER_CLIP, MIC_RATE)
    out_path = AUDIO_DIR / f"student_{STUDENT_ID}_utt_{i}.wav"
    sf.write(out_path, audio, MIC_RATE, subtype="PCM_16")
    peak = float(np.abs(audio).max())
    rms  = float(np.sqrt(np.mean(audio**2)))
    recording_log.append({
        "clip":     i,
        "filename": out_path.name,
        "duration_s": SECONDS_PER_CLIP,
        "peak":     round(peak, 3),
        "rms":      round(rms, 4),
        "size_kb":  round(out_path.stat().st_size / 1024, 1),
    })
    print(f"    saved {out_path.name}  (peak={peak:.2f}, rms={rms:.3f})")

    if i < N_CLIPS:
        print(f"    pause 3 seconds before clip {i+1}...")
        time.sleep(3)

# 5) Summary table
print(f"\nEnrollment recording complete for: {STUDENT_NAME}\n")
df_recording = pd.DataFrame(recording_log)
df_recording

## Extract features for your voice and find yourself on the map

Same pipeline as before — no new code, just applied to your three recordings. After this cell runs, you'll have:

- 3 new MFCC matrices in `features/`
- 3 new WavLM embeddings in `embeddings/`

Then we re-plot the cosine-similarity heatmap. You should see an **eleventh 3×3 bright block on the diagonal**: that's **you**, clustered against yourself, sitting at some distance from the ten LibriSpeech speakers.

How close or how far you land from any given LibriSpeech speaker has no biological meaning — it just reflects superficial acoustic similarities (pitch range, recording conditions, microphone). What matters is the *intra-block* brightness: your three utterances should cluster tightly with each other.

In [ ]:
# Extract MFCC + WavLM embedding for the student's three recordings
student_log = []
for wav_path in sorted(AUDIO_DIR.glob(f"student_{STUDENT_ID}_*.wav")):
    # MFCC
    mfcc = audio_to_mfcc(wav_path)
    np.save(FEATURES_DIR / f"{wav_path.stem}.npy", mfcc)

    # Embedding
    emb = wav_to_embedding(wav_path)
    np.save(EMBEDDINGS_DIR / f"{wav_path.stem}.npy", emb)

    student_log.append({
        "filename":    wav_path.name,
        "mfcc_shape":  str(mfcc.shape),
        "emb_dim":     emb.shape[0],
        "norm":        round(float(np.linalg.norm(emb)), 4),
    })

df_student = pd.DataFrame(student_log)
print(f"Voice bank now contains {len(SELECTED_SPEAKERS) + 1} identities: "
      f"{len(SELECTED_SPEAKERS)} LibriSpeech speakers + {STUDENT_NAME}\n")
df_student

In [ ]:
# Updated similarity heatmap — LibriSpeech speakers + the student
embedding_files = sorted(EMBEDDINGS_DIR.glob("*.npy"))
labels    = [f.stem for f in embedding_files]
speakers  = [lbl.split("_")[1] for lbl in labels]
E         = np.stack([np.load(f) for f in embedding_files])
S         = E @ E.T

fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(S, cmap="viridis", vmin=-0.2, vmax=1.0)

short = [f"{s}·u{lbl.split('_')[3]}" for s, lbl in zip(speakers, labels)]
ax.set_xticks(range(len(labels))); ax.set_xticklabels(short, rotation=90, ha="center", fontsize=8)
ax.set_yticks(range(len(labels))); ax.set_yticklabels(short, fontsize=8)

# Block separators between speakers — red, computed from data
unique_speakers = []
for s in speakers:
    if s not in unique_speakers:
        unique_speakers.append(s)
boundaries = np.cumsum([speakers.count(s) for s in unique_speakers])[:-1]
for k in boundaries:
    ax.axhline(k - 0.5, color="red", lw=1.2)
    ax.axvline(k - 0.5, color="red", lw=1.2)

# Highlight the student's block in cyan so it pops visually
student_indices = [i for i, s in enumerate(speakers) if s == STUDENT_ID]
if student_indices:
    s_start, s_end = min(student_indices) - 0.5, max(student_indices) + 0.5
    for edge in [s_start, s_end]:
        ax.axhline(edge, color="cyan", lw=2.5)
        ax.axvline(edge, color="cyan", lw=2.5)

ax.set_title(f"Voice bank — cosine similarity\n"
             f"({len(unique_speakers)} identities: 10 LibriSpeech + {STUDENT_NAME} — cyan = you)")
fig.colorbar(im, ax=ax, label="cosine similarity")
plt.tight_layout()
plt.show()

# Quantitative summary now that you're in the bank
same_spk_mask = np.array([[speakers[i] == speakers[j] for j in range(len(labels))]
                          for i in range(len(labels))])
off_diag  = ~np.eye(len(labels), dtype=bool)
same_vals = S[same_spk_mask & off_diag]
diff_vals = S[~same_spk_mask & off_diag]

print(f"Same-speaker pairs      (n={len(same_vals):>3}): mean={same_vals.mean():.3f}  min={same_vals.min():.3f}  max={same_vals.max():.3f}")
print(f"Different-speaker pairs (n={len(diff_vals):>3}): mean={diff_vals.mean():.3f}  min={diff_vals.min():.3f}  max={diff_vals.max():.3f}")
print(f"Gap                                    : {same_vals.min() - diff_vals.max():+.3f}")

# Your own consistency: how similar are your 3 clips to each other?
if student_indices:
    own_block = S[np.ix_(student_indices, student_indices)]
    own_off_diag = own_block[~np.eye(len(student_indices), dtype=bool)]
    print(f"\nYour intra-recording consistency (3 clips of yourself):")
    print(f"  mean cosine = {own_off_diag.mean():.4f}   min = {own_off_diag.min():.4f}   max = {own_off_diag.max():.4f}")
    print(f"  (healthy enrollment: mean > 0.95)")

## Recap — what we've built so far

Before we move into the database, let's pause and look at what's already on disk.

### The pipeline you just ran

```
raw .wav  →  load + trim silence  →  MFCC (n_mfcc=40)   →  features/*.npy
                                  ↘  WavLM encoder       →  embeddings/*.npy   (512-dim, L2-normalised)
```

### What you have right now

| Artefact | Location | Count | What it is |
|---|---|---|---|
| Raw audio clips | `audio/` | 33 files | 10 LibriSpeech speakers × 3 + you × 3 |
| MFCC matrices | `features/` | 33 files | `(40, n_frames)` — human-inspectable spectral features |
| **Speaker embeddings** | `embeddings/` | 33 files | `(512,)` — biometric templates, the production-grade representation |

### What we proved

1. **WavLM clusters voices by identity.** The block-diagonal pattern in the 33×33 cosine-similarity heatmap is unambiguous.
2. **Your own enrollment is healthy.** Your three clips matched each other at cosine ≈ 0.99 — well above the 0.95 threshold for trustworthy enrollment.
3. **A single global threshold is not enough.** The gap between same-speaker minimum and different-speaker maximum is slightly **negative** (≈ -0.03). Some impostor pairs in our 10-speaker bank score higher than the worst genuine pair. This is exactly the problem that calibrated thresholds (EER, FAR=1%) solve in the next section.

### What we'll do next

Persist these 33 embeddings into a **SQLite biometrics database** with one row per identity, calibrate a verification threshold from the same/different distributions, and turn it into a `verify()` function that can authenticate a new voice sample against the bank.

## Part 3 — Persisting voices to a biometric database

Up to now everything lives in scattered `.npy` files. That's fine for a notebook but terrible for a real system: no metadata, no atomic updates, no queries. Production voice biometrics systems store one row per enrolled identity in a real database.

### Schema design

We use **SQLite** because it ships with Python (zero install), and biometrics doesn't need the scale of Postgres for a workshop. The schema is intentionally minimal:

| Column | Type | What it stores |
|---|---|---|
| `id` | INTEGER PRIMARY KEY | Auto-incremented row ID |
| `name` | TEXT UNIQUE | Display name (`"John Doe"`, `"Federico"`) — what we show on a successful match |
| `source` | TEXT | `"librispeech"` or `"student"` — useful for debugging and demos |
| `enrolled_at` | TIMESTAMP | When this identity was added |
| `n_samples` | INTEGER | How many utterances were averaged into the embedding (3 in our case) |
| `embedding` | BLOB | The 512-dim float32 vector, L2-normalised, as raw bytes |

### Why one row per identity, not per utterance

In Part 1 we had 30 LibriSpeech embeddings (3 per speaker). For verification we collapse them into **one centroid per speaker** — the average of the three L2-normalised vectors, re-normalised. This is the standard enrollment trick: averaging cancels per-utterance noise (which utterance you said, what background sound was in the room) while preserving speaker identity (which is consistent across utterances). It also makes the verify step a single dot product instead of an aggregation.

The raw per-utterance `.npy` files stay on disk for inspection. The **database stores only the consolidated biometric template** — that's what gets queried at verification time.

In [ ]:
# SQLite schema for the biometrics database

# Drop any pre-existing DB so we always start from a known state in workshops
if DB_PATH.exists():
    DB_PATH.unlink()
    print(f"Removed existing DB at {DB_PATH}")

conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()

cur.executescript("""
CREATE TABLE voice_identities (
    id           INTEGER PRIMARY KEY AUTOINCREMENT,
    name         TEXT    UNIQUE NOT NULL,
    source       TEXT    NOT NULL CHECK (source IN ('librispeech', 'student')),
    enrolled_at  TIMESTAMP NOT NULL,
    n_samples    INTEGER NOT NULL,
    embedding    BLOB    NOT NULL
);

CREATE INDEX idx_source ON voice_identities(source);
""")
conn.commit()
print(f"Database created at {DB_PATH}")

# Verify schema
cur.execute("PRAGMA table_info(voice_identities);")
schema_df = pd.DataFrame(
    cur.fetchall(),
    columns=["col_id", "name", "type", "notnull", "default", "pk"]
)
print("\nSchema of voice_identities:")
schema_df

### Enrolling the 10 LibriSpeech speakers with fake names

LibriSpeech speakers are identified by anonymous integers (`1089`, `1188`, etc.). For the verification demo to feel real, we map each integer to a fictional human name. This is what a real enrollment system would store — names, not IDs.

The mapping is **deterministic and reproducible**: we pair the alphabetically-sorted speaker IDs with a fixed list of names. Every student running this notebook gets the same `1089 → "Aaron Brooks"` assignment.

In [ ]:
# Enrol the 10 LibriSpeech speakers as fictional identities
# Fixed mapping: alphabetically-sorted speaker IDs -> fictional names
LIBRISPEECH_NAMES = [
    "Aaron Brooks",
    "Beatriz Cordero",
    "Carlos Diaz",
    "Diana Esposito",
    "Ethan Fischer",
    "Fatima Gallo",
    "Gabriel Huang",
    "Hannah Ibarra",
    "Ivan Johansson",
    "Julia Kowalski",
]
assert len(LIBRISPEECH_NAMES) == len(SELECTED_SPEAKERS), "Name list length mismatch"

def build_centroid(speaker_id: str) -> np.ndarray:
    """Average the 3 utterance embeddings for one speaker and re-normalise."""
    files = sorted(EMBEDDINGS_DIR.glob(f"speaker_{speaker_id}_utt_*.npy"))
    embs  = np.stack([np.load(f) for f in files])
    centroid = embs.mean(axis=0)
    return centroid / np.linalg.norm(centroid)

# Insert one row per speaker
now = datetime.utcnow().isoformat()
enrolled_log = []
for spk_id, display_name in zip(SELECTED_SPEAKERS, LIBRISPEECH_NAMES):
    centroid = build_centroid(spk_id)
    cur.execute(
        "INSERT INTO voice_identities (name, source, enrolled_at, n_samples, embedding) "
        "VALUES (?, ?, ?, ?, ?)",
        (display_name, "librispeech", now, 3, centroid.astype(np.float32).tobytes())
    )
    enrolled_log.append({
        "speaker_id":   spk_id,
        "display_name": display_name,
        "centroid_dim": centroid.shape[0],
        "centroid_norm": round(float(np.linalg.norm(centroid)), 4),
    })

conn.commit()
print(f"Enrolled {len(enrolled_log)} LibriSpeech identities into the database.\n")

df_enrolled = pd.DataFrame(enrolled_log)
df_enrolled

### Enrolling yourself

Same procedure, different source. Your three utterances become a centroid, the centroid lands in the same table — just with `source='student'` and your real name.

If a student with the same name already exists in the DB (you're re-running the cell), we overwrite the previous enrollment rather than fail. In a production system this would be a careful re-enrollment flow with audit logging; here we just `INSERT OR REPLACE`.

In [ ]:
# Enrol the student into the same table
student_files = sorted(EMBEDDINGS_DIR.glob(f"student_{STUDENT_ID}_utt_*.npy"))
if len(student_files) != 3:
    raise RuntimeError(
        f"Expected 3 enrollment embeddings for {STUDENT_NAME}, found {len(student_files)}. "
        f"Re-run the recording cell."
    )

student_embs     = np.stack([np.load(f) for f in student_files])
student_centroid = student_embs.mean(axis=0)
student_centroid = student_centroid / np.linalg.norm(student_centroid)

now = datetime.utcnow().isoformat()
cur.execute(
    "INSERT OR REPLACE INTO voice_identities (name, source, enrolled_at, n_samples, embedding) "
    "VALUES (?, ?, ?, ?, ?)",
    (STUDENT_NAME, "student", now, len(student_files),
     student_centroid.astype(np.float32).tobytes())
)
conn.commit()

print(f"Enrolled '{STUDENT_NAME}' into the database.")
print(f"  centroid dim  : {student_centroid.shape[0]}")
print(f"  centroid norm : {np.linalg.norm(student_centroid):.4f}")
print(f"  n_samples     : {len(student_files)}")

### Inspecting the database

Let's query the table. This is what the system "knows" — eleven identities, each represented by a single 2 KB blob.

In [ ]:
# Show all enrolled identities as a clean table
query = """
SELECT
    id,
    name,
    source,
    n_samples,
    LENGTH(embedding) AS embedding_bytes,
    enrolled_at
FROM voice_identities
ORDER BY id DESC;
"""

df_db = pd.read_sql_query(query, conn)
print(f"voice_identities — {len(df_db)} rows enrolled\n")
df_db

### What does a 512-dimensional voice fingerprint actually look like?

Up to now we've talked about "embeddings" as abstract vectors. Let's look at the raw numbers.

Each row in the database is a `BLOB` of **2048 bytes = 512 × 4 bytes = 512 float32 numbers**, L2-normalised (every row's vector has length exactly 1.0). To a human these are just opaque numbers — there's no "this dimension encodes pitch, this one encodes accent." The meaning is entirely **geometric**: two voices are the same person if their vectors point in similar directions in this 512-dim space.

We'll inspect them three ways:

1. **A numeric preview** — first 8 dimensions of each identity, as a plain table.
2. **A heatmap of the full bank** — 11 rows × 512 columns, every value colour-coded.
3. **One single embedding as a line plot** — to give a feel for how erratic and "spiky" a real biometric template looks.

In [ ]:
# Visualise the raw embeddings stored in the database
# 1) Pull every embedding back out of the DB and decode bytes -> np.float32
rows = cur.execute(
    "SELECT id, name, embedding FROM voice_identities ORDER BY id ASC;"
).fetchall()

ids       = [r[0] for r in rows]
names     = [r[1] for r in rows]
emb_matrix = np.stack([
    np.frombuffer(r[2], dtype=np.float32) for r in rows
])  # shape: (11, 512)

print(f"Loaded embedding matrix from DB: shape={emb_matrix.shape}, dtype={emb_matrix.dtype}")
print(f"All rows L2-normalised? norms = {np.linalg.norm(emb_matrix, axis=1).round(4)}\n")

# 2) Numeric preview — first 8 dimensions per identity
preview = pd.DataFrame(
    emb_matrix[:, :8],
    columns=[f"dim_{i}" for i in range(8)],
    index=[f"{i}. {n}" for i, n in zip(ids, names)],
).round(4)
print("First 8 dimensions of each enrolled identity:")
preview

In [ ]:
# Full 11×512 heatmap + one identity as a line plot
fig, axes = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={"height_ratios": [2, 1]})

# Top: heatmap of all 11 embeddings × 512 dimensions
im = axes[0].imshow(emb_matrix, cmap="RdBu_r", aspect="auto", vmin=-0.15, vmax=0.15)
axes[0].set_yticks(range(len(names)))
axes[0].set_yticklabels([f"{i}. {n}" for i, n in zip(ids, names)], fontsize=9)
axes[0].set_xlabel("Embedding dimension (0 – 511)")
axes[0].set_title("All 11 enrolled voice fingerprints — each row is one identity")
fig.colorbar(im, ax=axes[0], label="value")

# Bottom: one identity as a 1D line — show the spikiness of a single template
target_idx  = len(rows) - 1                    # last enrolled = the student
target_name = names[target_idx]
axes[1].plot(emb_matrix[target_idx], lw=0.6, color="steelblue")
axes[1].set_xlabel("Embedding dimension")
axes[1].set_ylabel("Value")
axes[1].set_title(f"A single voice fingerprint as a 1D signal — {target_name}")
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_xlim(0, 511)

plt.tight_layout()
plt.show()

print(f"\nKey takeaways:")
print(f"  - Every row in the heatmap is visibly different. That difference is the speaker identity.")
print(f"  - The line plot is spiky and irregular. No human could 'design' this by hand.")
print(f"  - All these vectors live on a unit sphere in 512-D space (norm = 1).")
print(f"  - Verification = checking if a new vector points in roughly the same direction as a stored one.")

## Part 4 — Calibrating the verification threshold

We have 11 enrolled identities. To actually **authenticate** a new voice sample, we need to answer: "is this person who they claim to be?" That decision boils down to a single number — the **threshold** above which we accept the claim and below which we reject it.

### Two scores, two distributions

Take every pair of utterances we have on disk (33 utterances → 528 unordered pairs). Each pair falls into one of two categories:

- **Genuine pair** — both utterances come from the same speaker. We want these to score *high*.
- **Impostor pair** — utterances come from different speakers. We want these to score *low*.

If WavLM is a good encoder, these two distributions should be visibly separated. Where to place the threshold *inside* that separation is the calibration problem.

### Two standard threshold choices

| Metric | Definition | When to use |
|---|---|---|
| **EER** (Equal Error Rate) | Threshold where False Accept Rate (FAR) = False Reject Rate (FRR) | Academic benchmark — the "neutral" point of the system |
| **FAR = 1%** | Threshold where only 1% of impostor pairs are wrongly accepted | **Production biometrics** — banking, device unlock, anything where letting an impostor in is worse than rejecting a genuine user |

For our workshop we report both and use **FAR=1%** for the actual verifier. Voice biometrics is asymmetric: an impostor breaking in is a security incident; a legitimate user getting rejected is a minor annoyance fixed by re-trying.

In [ ]:
# Calibrate the verification threshold from genuine vs impostor pair distributions

# 1) Load all 33 raw utterance embeddings (not centroids — we want per-utterance scores)
all_emb_files = sorted(EMBEDDINGS_DIR.glob("*.npy"))
all_labels    = [f.stem for f in all_emb_files]          # e.g. "speaker_1089_utt_1"
all_speakers  = [lbl.split("_")[1] for lbl in all_labels]
all_embs      = np.stack([np.load(f) for f in all_emb_files])
print(f"Loaded {len(all_embs)} utterance embeddings.")

# 2) Compute every unordered pair, label each as genuine or impostor
pairs = []
for i, j in combinations(range(len(all_embs)), 2):
    score = float(all_embs[i] @ all_embs[j])             # cosine sim (already L2-normalised)
    is_genuine = (all_speakers[i] == all_speakers[j])
    pairs.append({"i": i, "j": j, "score": score, "genuine": is_genuine})

df_pairs = pd.DataFrame(pairs)
n_genuine  = df_pairs["genuine"].sum()
n_impostor = len(df_pairs) - n_genuine
print(f"  Genuine pairs  : {n_genuine}")
print(f"  Impostor pairs : {n_impostor}\n")

# 3) Sweep candidate thresholds, compute FAR and FRR at each
thresholds = np.linspace(0.0, 1.0, 1001)
genuine_scores  = df_pairs.loc[df_pairs["genuine"], "score"].values
impostor_scores = df_pairs.loc[~df_pairs["genuine"], "score"].values

far = np.array([(impostor_scores >= t).mean() for t in thresholds])   # impostor accepted
frr = np.array([(genuine_scores  <  t).mean() for t in thresholds])   # genuine rejected

# 4) EER: minimise |FAR - FRR|
eer_idx       = np.argmin(np.abs(far - frr))
eer_threshold = thresholds[eer_idx]
eer_rate      = (far[eer_idx] + frr[eer_idx]) / 2

# 5) FAR=1%: smallest threshold where FAR <= 0.01
far1_idx       = np.argmax(far <= 0.01)                  # first True
far1_threshold = thresholds[far1_idx]
far1_frr       = frr[far1_idx]

print(f"Calibration results:")
print(f"  EER       : {eer_rate*100:.2f}%   at threshold = {eer_threshold:.4f}")
print(f"  FAR = 1%  : FRR = {far1_frr*100:.2f}%  at threshold = {far1_threshold:.4f}")
print(f"\nWe will use threshold = {far1_threshold:.4f} for the verifier (FAR=1% operating point).")

# Persist for use in later cells
ACCEPT_THRESHOLD = far1_threshold

### Visualising the calibration

Two plots make the math feel concrete:

1. **The two distributions** — genuine and impostor scores as histograms. The visible overlap is where errors come from; the threshold cuts through that overlap.
2. **The DET-style curve** — FAR vs FRR as a function of threshold. EER is where they cross. FAR=1% is the conservative operating point we'll actually use.

In [ ]:
# Visualise the score distributions and the FAR/FRR curves
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: histograms of genuine vs impostor scores
axes[0].hist(impostor_scores, bins=40, alpha=0.6, label=f"Impostor (n={len(impostor_scores)})",
             color="crimson", edgecolor="black")
axes[0].hist(genuine_scores,  bins=40, alpha=0.6, label=f"Genuine (n={len(genuine_scores)})",
             color="seagreen", edgecolor="black")
axes[0].axvline(eer_threshold,  color="orange", linestyle="--", lw=2,
                label=f"EER threshold = {eer_threshold:.3f}")
axes[0].axvline(far1_threshold, color="blue", linestyle="--", lw=2,
                label=f"FAR=1% threshold = {far1_threshold:.3f}")
axes[0].set_xlabel("Cosine similarity score")
axes[0].set_ylabel("Number of pairs")
axes[0].set_title("Score distributions — overlap zone is where errors live")
axes[0].legend(loc="upper left")

# Right: FAR and FRR vs threshold
axes[1].plot(thresholds, far*100, label="FAR (impostor accepted)", color="crimson", lw=2)
axes[1].plot(thresholds, frr*100, label="FRR (genuine rejected)",  color="seagreen", lw=2)
axes[1].axvline(eer_threshold,  color="orange", linestyle="--", lw=2,
                label=f"EER = {eer_rate*100:.2f}%")
axes[1].axvline(far1_threshold, color="blue", linestyle="--", lw=2,
                label=f"FAR=1% operating point")
axes[1].set_xlabel("Threshold")
axes[1].set_ylabel("Error rate (%)")
axes[1].set_title("FAR and FRR as a function of threshold")
axes[1].set_xlim(0.3, 1.0)
axes[1].set_ylim(0, 50)
axes[1].legend()
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

print(f"Production operating point:")
print(f"  Threshold = {ACCEPT_THRESHOLD:.4f}")
print(f"  Expected FAR ~ 1%   (≈1 impostor in 100 wrongly accepted)")
print(f"  Expected FRR ~ {far1_frr*100:.1f}%   (≈{far1_frr*100:.0f} genuine users in 100 wrongly rejected — they re-try)")

### Reading the two plots

**Left — score distributions.** Every pair of utterances in our 33-clip bank gets a cosine-similarity score. Pairs from the *same* speaker (green, n=33) cluster near 1.0; pairs from *different* speakers (red, n=495) spread across a wide range with most of the mass between 0.6 and 0.9. The threshold is the vertical line that decides which side of the cut counts as "accept." Errors come from the **overlap zone** — the small region where some genuine pairs score below some impostor pairs. There is no threshold that eliminates the overlap; you can only choose which kind of error to make.

**Right — what happens when you move the threshold.** Each point on the X axis is a *candidate threshold*. For every candidate we compute two error rates over the 528 already-labelled pairs:

- **FAR — False Accept Rate** (red curve). Of the 495 impostor pairs, what fraction score *above* this threshold and get wrongly accepted as "same person"? A low threshold is permissive → many impostors slip through → FAR high. A high threshold is strict → fewer impostors pass → FAR low.

- **FRR — False Reject Rate** (green curve). Of the 33 genuine pairs, what fraction score *below* this threshold and get wrongly rejected as "different person"? A low threshold accepts everyone → FRR ≈ 0. A high threshold rejects even legitimate users → FRR rises.

The two curves always move in opposite directions. **Raising the threshold lowers FAR but raises FRR.** You cannot reduce both at the same time — this is the irreducible trade-off of any biometric authentication system.

**The two vertical lines:**

- **Orange (EER)** — the threshold where the two curves cross. FAR = FRR = 3.03%. It's the mathematically "neutral" point, useful as an academic benchmark.
- **Blue (FAR=1%)** — slightly to the right. We force FAR ≤ 1% (stricter against impostors) and accept that FRR rises to 3%. This is the threshold we'll use in production.

**The closing question — which error do you prefer to make?**

Choosing between EER and FAR=1% isn't a math question, it's a policy question. **Which is worse: letting an impostor in, or rejecting a legitimate user?**

- A bank unlocking accounts by voice → impostor entry is a security breach. Pick **FAR=1%** or stricter (FAR=0.1%, FAR=0.01% for high-value transactions). The legitimate user can re-try.
- A smart speaker recognizing family members for personalized greetings → false rejection is annoying and breaks UX. Pick something closer to **EER** or even more permissive.

There is no universally correct threshold. There is only a threshold appropriate to the cost asymmetry of *your* application.

## Part 5 — The `verify()` function

Everything so far has been preparation. This is the function the alumni will use in the battle demo: feed it a `.wav` file, and it answers two questions:

1. **Is the speaker enrolled in the database?** (returns a name or "no match")
2. **How confident is the match?** (returns a similarity score and the top-3 closest identities)

The logic is simple — extract the embedding from the new audio, compute cosine similarity against every row in the DB, take the argmax, and accept the match only if the score clears `ACCEPT_THRESHOLD` (our FAR=1% calibration).

In [ ]:
# The verify() function: authenticate a voice clip against the enrolled DB
@dataclass
class VerificationResult:
    matched:         bool
    matched_name:    Optional[str]
    matched_score:   float
    threshold:       float
    top3:            list   # list of (name, score) tuples

    def __repr__(self):
        verdict = f"✅ MATCH: {self.matched_name}" if self.matched else "❌ NO MATCH"
        return (f"<{verdict}  score={self.matched_score:.4f}  "
                f"threshold={self.threshold:.4f}>")


def load_db_embeddings(conn: sqlite3.Connection):
    """Pull every row from the DB. Returns (names, source, embedding_matrix)."""
    rows = conn.execute(
        "SELECT name, source, embedding FROM voice_identities ORDER BY id ASC;"
    ).fetchall()
    names    = [r[0] for r in rows]
    sources  = [r[1] for r in rows]
    matrix   = np.stack([np.frombuffer(r[2], dtype=np.float32) for r in rows])
    return names, sources, matrix


def verify(wav_path: Path,
           conn:      sqlite3.Connection,
           threshold: float = ACCEPT_THRESHOLD) -> VerificationResult:
    """
    Authenticate the speaker in `wav_path` against every enrolled identity.

    Returns a VerificationResult with the best match (if any clears the threshold)
    and the top-3 candidates regardless of threshold (useful for debugging).
    """
    # 1) Extract embedding from the new audio (L2-normalised by wav_to_embedding)
    probe = wav_to_embedding(wav_path)

    # 2) Pull all enrolled centroids and compute cosine similarity in one shot
    names, _, db_matrix = load_db_embeddings(conn)
    scores = db_matrix @ probe                 # shape (N,)

    # 3) Rank and pick top-3
    order  = np.argsort(scores)[::-1]
    top3   = [(names[i], float(scores[i])) for i in order[:3]]

    best_name, best_score = top3[0]
    matched = best_score >= threshold

    return VerificationResult(
        matched       = matched,
        matched_name  = best_name if matched else None,
        matched_score = best_score,
        threshold     = threshold,
        top3          = top3,
    )


# Smoke test: every utterance should match its own enrolled identity
print("Smoke test — verifying each enrollment utterance against the full DB:\n")
test_log = []
for wav_path in sorted(AUDIO_DIR.glob("*.wav")):
    # Resolve the "true" name from filename
    if wav_path.stem.startswith("speaker_"):
        spk_id   = wav_path.stem.split("_")[1]
        true_idx = SELECTED_SPEAKERS.index(spk_id)
        true_name = LIBRISPEECH_NAMES[true_idx]
    else:
        true_name = STUDENT_NAME

    result = verify(wav_path, conn)
    test_log.append({
        "file":          wav_path.name,
        "true_identity": true_name,
        "matched":       result.matched,
        "predicted":     result.matched_name or "(rejected)",
        "score":         round(result.matched_score, 4),
        "correct":       result.matched and result.matched_name == true_name,
    })

df_smoke = pd.DataFrame(test_log)
correct  = df_smoke["correct"].sum()
total    = len(df_smoke)
print(f"Self-verification accuracy: {correct}/{total} ({correct/total*100:.1f}%)\n")
df_smoke

## Part 6 — The battle demo: can your voice fool the system?

Time to actually attack the database. The smoke test we just ran was easy mode — we verified utterances that were already used during enrollment, so of course they matched. Now we run the harder test: **a brand-new recording**, never seen during enrollment, has to win against all 11 enrolled identities.

### Two scenarios to run with the class

1. **Genuine attempt** — *you* record a fresh clip (different sentence, different mood, even different day) and try to verify against your own enrollment. Should pass.
2. **Impostor attempt** — *a classmate* records a fresh clip claiming to be you. Should fail.

Both scenarios use the **exact same `verify()` function** — the system doesn't "know" who's claiming what. It just compares the probe embedding against every enrolled centroid and picks the closest one. Identity claims are not part of the verification math; the system tells you *who you sound like*, not whether you are who you say you are.

In [ ]:
# Battle demo: record a fresh clip and verify against the DB
print("=" * 70)
print("BATTLE DEMO — fresh recording vs enrolled database")
print("=" * 70)
print("\nRecord a NEW clip — not used during enrollment. ~6 seconds is enough.")
print("Try variations to see what fools the system: whisper, shout, foreign accent,")
print("or have a classmate try to impersonate you.\n")

BATTLE_CLIP_SECONDS = 6
input("Press ENTER when ready to record... ")

print(">>> Recording — speak NOW...")
audio = record_clip(BATTLE_CLIP_SECONDS, MIC_RATE)

# Save with timestamp so multiple attempts don't overwrite each other
battle_path = AUDIO_DIR / f"battle_{int(time.time())}.wav"
sf.write(battle_path, audio, MIC_RATE, subtype="PCM_16")
print(f"Saved {battle_path.name}\n")

# Run verification
result = verify(battle_path, conn)

# Pretty-print the verdict
print("=" * 70)
if result.matched:
    print(f"  ✅  AUTHENTICATED  →  {result.matched_name}")
    print(f"      score = {result.matched_score:.4f}  ≥  threshold = {result.threshold:.4f}")
else:
    print(f"  ❌  ACCESS DENIED  —  no enrolled identity matched")
    print(f"      best score = {result.matched_score:.4f}  <  threshold = {result.threshold:.4f}")
    print(f"      closest enrolled identity (rejected): {result.top3[0][0]}")
print("=" * 70)

# Show top-3 ranking regardless of outcome
print("\nTop-3 closest enrolled identities:")
df_top3 = pd.DataFrame(result.top3, columns=["identity", "cosine_similarity"])
df_top3["above_threshold"] = df_top3["cosine_similarity"] >= ACCEPT_THRESHOLD
df_top3

### Reading the top-3 ranking

The `top3` view is what makes failures interpretable. When a genuine attempt fails (you = rejected), the top-3 still puts you first — you're just below threshold. That's a FRR event (false reject), fixable by re-trying.

When an impostor succeeds (classmate = accepted as you), it means their voice landed close enough to your enrolled centroid that the system can't distinguish them. That's a FAR event (false accept), and it's the one we engineered the threshold to keep below 1%.

When **multiple** people in your class match the same enrolled identity above threshold, you've found a real-world manifestation of the impostor cluster: some voices in your DB are genuinely indistinguishable to WavLM at this resolution. The fix in a production system isn't to lower the threshold — it's to **demand multi-factor authentication** (voice + passphrase, voice + device fingerprint) for ambiguous cases.

### When the system gets it wrong — what just happened

If your first battle attempt failed (matched the wrong identity, or you matched yourself but barely), congratulations: you just witnessed the most important lesson of voice biometrics.

**Voice embeddings encode "how a voice sounds in a recording", not "who a person biologically is."** The model captures:
- Timbre and pitch range (your physical larynx) → consistent
- Speaking style, prosody, accent → mostly consistent
- Microphone characteristics, room acoustics, recording level → **variable per session**
- Distance to mic, background noise, time of day → **highly variable**

When your enrollment was recorded under one set of conditions and your battle clip under another, the embedding moves — sometimes by more than the distance to other enrolled speakers. This is not a bug in WavLM, it is the **fundamental brittleness of session-dependent biometrics**.

**This is exactly why production voice authentication systems do not work the way we just built it.** Real systems use:

1. **Multi-session enrollment** — collect samples across days, devices, and conditions to build a robust centroid that covers natural variation.
2. **Channel/noise compensation** — apply VAD, denoising, and bandwidth normalisation before embedding extraction.
3. **Text-dependent verification** — ask the user to say a specific passphrase ("My voice is my password") so the linguistic content is also part of the match.
4. **Multi-factor authentication** — voice alone is never enough for high-value actions. Always combined with device, PIN, or behavioural signals.

The system you built today is a **proof of concept**, not a production authenticator. The 1% FAR we calibrated holds **only inside the controlled distribution of LibriSpeech-quality audio**. The moment you move outside that distribution (your laptop mic, a different room, a different mood), all bets are off.

This is the honest answer to "can I unlock my bank account with my voice?" — yes, but only because the bank does ten things we are not doing here.

### What you (probably) just learned

Two battle attempts in a row with the **same speaker** and the **same microphone** likely gave you very different verification scores — possibly different verdicts altogether — depending on what you said.

The biological identity didn't change between attempts. The recording conditions barely did. What changed was the **linguistic content** — and that was enough to move your embedding by a meaningful amount in 512-D space, sometimes more than the distance to other enrolled speakers.

This is why every production voice biometrics system on the market today is **text-dependent**: the user is asked to say a specific phrase during both enrollment and verification. It is not a UX choice, it is a mathematical necessity to keep the embedding distance stable across sessions.

## Part 7 — Deepfake detection and audio watermarking

You have a working voice biometrics system. The closing question of this workshop is: **how does it hold up against synthetic audio?**

This part has two halves:

### 7a — Synthetic audio vs the biometric database

We generate fake audio using a **public text-to-speech (TTS) service** — `gTTS`, Google's free TTS API. The synthetic clips don't try to clone any specific person's voice; they're generated with Google's default neutral voice. Then we run them through `verify()` and see what happens.

There are two possible outcomes, both interesting:

1. **The fake gets rejected** — good news: our system already filters out generic TTS audio because its acoustic profile is too far from any real human in the DB.
2. **The fake matches someone** — bad news: a generic TTS voice happened to land closer to one of our enrolled identities than 1% of impostor pairs do.

Either way, the lesson is the same: a verification system built only on a similarity threshold is not enough. We need a **separate detector** that can answer the question *"was this audio generated by a machine?"* — independently of *"whose voice does it sound like?"*

### 7b — Watermarking: building detection into the audio itself

After we see how the biometric system reacts to synthetic audio, we look at the modern defence: **watermarking**. The idea is to embed an imperceptible signal into TTS-generated audio at the moment of generation, so it can be detected later — even after compression, noise, or re-recording.

We do this two ways:

- **Classical watermarking** — modulating specific frequency bins in the spectrogram. Fully manual, fully transparent. You'll see the watermark with your own eyes in the spectrogram.
- **Neural watermarking with AudioSeal (Meta, 2024)** — the production-grade version. ~50 MB model, designed specifically for marking TTS output and surviving real-world audio manipulation.

### 7a-1 — Loading pre-generated synthetic audio

Generating TTS audio reliably across Windows, macOS, and Linux laptops in real-time during a workshop turned out to be a deployment headache (we tried `pyttsx3` and it stalled on Windows). So we did the pragmatic thing: **the three deepfake clips were generated once with Microsoft SAPI5 (the native TTS engine of Windows) and committed to the workshop repo**.

If you want to reproduce the generation yourself on Windows, here is the PowerShell snippet that produced these files:

```powershell
Add-Type -AssemblyName System.Speech
$synth = New-Object System.Speech.Synthesis.SpeechSynthesizer
$synth.SetOutputToWaveFile("deepfakes\fake_generic.wav")
$synth.Speak("Hello, this is a test of the voice authentication system.")
$synth.Dispose()
```

For our purposes the source of the synthetic audio doesn't matter — what matters is that it's a clean, off-the-shelf TTS voice (no voice cloning, no fine-tuning) trying to bypass a real biometric system.

In [ ]:
# Load the pre-generated deepfake clips
DEEPFAKE_DIR = WORKSHOP_DIR / "deepfakes"

if not DEEPFAKE_DIR.exists() or not list(DEEPFAKE_DIR.glob("*.wav")):
    raise FileNotFoundError(
        f"No deepfake WAVs found in {DEEPFAKE_DIR}. "
        f"Make sure you pulled the deepfakes/ folder from the workshop repo."
    )

deepfake_log = []
for wav_path in sorted(DEEPFAKE_DIR.glob("*.wav")):
    info = sf.info(wav_path)
    deepfake_log.append({
        "filename":    wav_path.name,
        "duration_s":  round(info.duration, 2),
        "sample_rate": info.samplerate,
        "channels":    info.channels,
        "size_kb":     round(wav_path.stat().st_size / 1024, 1),
    })

df_fakes = pd.DataFrame(deepfake_log)
print(f"Loaded {len(df_fakes)} synthetic clips from {DEEPFAKE_DIR}\n")
df_fakes

### 7a-2 — Listen, then attack the biometric system

Before we hit the database, **listen to the three clips** — there's a player below each. The voice is robotic but clearly intelligible. No human would mistake it for a real person.

The question is whether **WavLM** would. The embedding space was trained on real human speech (VoxCeleb interviews) — synthetic audio is, in principle, outside the training distribution. So one of two things will happen:

1. **The fakes land far from every enrolled identity** (low cosine to everyone). The system correctly recognises that this audio is acoustically "weird" and rejects it. Win for the defender.
2. **The fakes accidentally land close to some enrolled identity** (cosine above threshold). Even though no human would mistake this voice for a real person, the embedding space *as a similarity metric* fails. Loss for the defender.

We run all three deepfake clips through `verify()` and see which scenario plays out.

In [ ]:
# Listen to each deepfake, then verify it against the biometric database
attack_log = []
for wav_path in sorted(DEEPFAKE_DIR.glob("*.wav")):
    print(f"\n{'='*70}")
    print(f"  {wav_path.name}")
    print(f"{'='*70}")
    display(Audio(str(wav_path)))

    result = verify(wav_path, conn)

    print(f"  Best match     : {result.top3[0][0]}")
    print(f"  Score          : {result.matched_score:.4f}")
    print(f"  Threshold      : {result.threshold:.4f}")
    print(f"  Verdict        : {'✅ ACCEPTED' if result.matched else '❌ REJECTED'}")
    print(f"  Top-3          :")
    for name, score in result.top3:
        flag = "  ←  ABOVE THRESHOLD" if score >= result.threshold else ""
        print(f"      {name:25s}  {score:.4f}{flag}")

    attack_log.append({
        "deepfake":   wav_path.name,
        "best_match": result.top3[0][0],
        "score":      round(result.matched_score, 4),
        "above_threshold": result.matched,
        "second_best":     result.top3[1][0],
        "second_score":    round(result.top3[1][1], 4),
    })

print(f"\n{'='*70}")
print("ATTACK SUMMARY")
print(f"{'='*70}\n")
df_attacks = pd.DataFrame(attack_log)
df_attacks

### What we just learned, and why we need watermarking

Three synthetic clips were all rejected. So we're safe, right?

**No.** Look at the table again. The best match for every single deepfake was a real enrolled identity (in this run, mostly *you*), at a cosine around 0.88. Your FAR=1% threshold is 0.919. **The margin between "rejected" and "accepted" is roughly 0.03** — the same margin that flipped your own battle attempts from success to failure when you changed the sentence.

What this means:

1. **WavLM does not detect "fakeness".** It detects "distance to the nearest enrolled human voice." Microsoft SAPI5 is rejected because its acoustic profile is far from any real voice in our database — not because the system recognised it as machine-generated. If we had used a modern voice-cloning TTS trained on *your* voice (XTTS, ElevenLabs, Resemble), the synthetic embedding would have landed near your centroid and passed the threshold.

2. **The defence we have is fragile by design.** A similarity-based verifier cannot distinguish "a real human who sounds like you" from "a machine that's been instructed to sound like you." Both produce embeddings near your centroid; both pass the threshold. The verifier has no extra channel of information.

3. **The fix is not a better threshold.** Lowering FAR to 0.1% would reject the fakes but also reject most of your own legitimate retries. Raising it would let humans through but let fakes in too. The shape of the score distributions doesn't separate synthetic from genuine — only different speakers.

**We need a separate signal**, embedded into the synthetic audio itself at the moment of generation, that survives compression and noise and can be detected later. This is **audio watermarking**, and it's what the rest of this notebook is about.

## 7b — Watermarking, part 1: classical frequency-domain marking

The core idea of audio watermarking is to **embed a hidden signal in the audio that doesn't change how it sounds, but can be detected by an algorithm that knows what to look for**.

### The classical trick: modulate inaudible frequency bins

Human hearing has known weak spots. We don't perceive small energy changes in **narrow frequency bands** within otherwise busy spectral regions. So the recipe is:

1. Take the audio's **spectrogram** (STFT — short-time Fourier transform).
2. Pick a frequency band the human ear barely uses — for speech, the band around **4–5 kHz** is a good target: still audible enough to survive compression, but quiet enough in normal speech that nudging it doesn't change perceived quality.
3. **Inject a known pattern** into that band — a sinusoidal "ping" at a specific frequency, or a binary code modulated as alternating energy levels.
4. Inverse-transform back to time domain. Save as WAV.

The result sounds **identical** to the unwatermarked original to a human listener, but the watermark is a clear deterministic pattern that a detector can recover from the spectrogram.

### What we'll do

Embed a 1-second tone at 4.5 kHz (very low amplitude) into one of our deepfake clips. Then in the detection step, we'll show:

1. The waveforms look identical to the naked eye.
2. The audio sounds identical to the ear.
3. The **spectrograms differ visibly** at the watermarked frequency.
4. A correlation-based detector finds the watermark with high confidence.

This is the simplest possible watermarking scheme — easy to break, easy to remove with a notch filter — but it makes the concept tangible before we move to a neural watermarker (AudioSeal) that's robust to real attacks.

In [ ]:
# Apply a classical frequency-band watermark to one of our deepfake clips

WATERMARK_DIR = WORKSHOP_DIR / "watermarked"
WATERMARK_DIR.mkdir(exist_ok=True)

# Watermark parameters
WM_FREQ_HZ    = 4500          # Hz — quiet zone for speech, survives compression
WM_AMPLITUDE  = 0.015         # tuned for reliable detection while staying inaudible
WM_DURATION_S = None          # None = mark the entire clip; or set a float in seconds

def apply_classical_watermark(
    input_wav:  Path,
    output_wav: Path,
    freq_hz:    float = WM_FREQ_HZ,
    amplitude:  float = WM_AMPLITUDE,
    duration_s: float = WM_DURATION_S,
) -> dict:
    """Embed a sinusoidal tone at freq_hz into the audio. Returns metadata."""
    audio, sr = sf.read(input_wav)
    n_samples = len(audio)

    t = np.arange(n_samples) / sr
    watermark = amplitude * np.sin(2 * np.pi * freq_hz * t)

    if duration_s is not None:
        cutoff = int(duration_s * sr)
        watermark[cutoff:] = 0.0

    watermarked = audio + watermark

    peak = np.abs(watermarked).max()
    if peak > 0.99:
        watermarked = watermarked * (0.99 / peak)

    sf.write(output_wav, watermarked, sr, subtype="PCM_16")

    audio_rms = np.sqrt(np.mean(audio**2))
    wm_rms    = np.sqrt(np.mean(watermark**2))
    snr_db    = 20 * np.log10(audio_rms / wm_rms) if wm_rms > 0 else float("inf")

    return {
        "input":          input_wav.name,
        "output":         output_wav.name,
        "sample_rate":    sr,
        "freq_hz":        freq_hz,
        "amplitude":      amplitude,
        "audio_rms":      round(float(audio_rms), 5),
        "watermark_rms":  round(float(wm_rms), 5),
        "snr_db":         round(float(snr_db), 1),
    }

SOURCE_FAKE      = DEEPFAKE_DIR / "fake_attack.wav"
WATERMARKED_FAKE = WATERMARK_DIR / "fake_attack_watermarked.wav"

meta = apply_classical_watermark(SOURCE_FAKE, WATERMARKED_FAKE)
print(f"Watermark applied:")
for k, v in meta.items():
    print(f"  {k:15s}: {v}")

print(f"\nThe watermarked audio is {meta['snr_db']:.1f} dB quieter than the speech itself")
print(f"— that's far below human audibility threshold for a tone in speech.\n")

print("Original:")
display(Audio(str(SOURCE_FAKE)))
print("Watermarked:")
display(Audio(str(WATERMARKED_FAKE)))

### Visualising the watermark

To the ear, the two clips are indistinguishable. The detection magic happens in the **frequency domain**. We plot:

1. **The waveforms** of both — should look identical because the watermark amplitude is 0.2% of full scale.
2. **The spectrograms** of both — should be visibly different at exactly 4.5 kHz, where we injected the tone.
3. **A zoomed-in spectrum slice at 4.5 kHz** — the smoking gun.

In [ ]:
# Visualise the watermark in time and frequency
orig_audio, sr = sf.read(SOURCE_FAKE)
wm_audio,   _  = sf.read(WATERMARKED_FAKE)

fig, axes = plt.subplots(3, 1, figsize=(14, 11))

# 1) Time-domain waveforms overlaid
t = np.arange(len(orig_audio)) / sr
axes[0].plot(t, orig_audio, color="steelblue", lw=0.5, label="Original")
axes[0].plot(t, wm_audio - orig_audio, color="crimson", lw=0.5,
             label="Watermark only (difference signal, amplified ×20 for visibility)",
             alpha=0.8)
# Re-plot watermark scaled up so it's visible
axes[0].plot(t, (wm_audio - orig_audio) * 20, color="crimson", lw=0.5, alpha=0.6)
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")
axes[0].set_title("Waveform comparison — watermark is invisible at true scale")
axes[0].legend(loc="upper right")

# 2) Spectrogram of both, side-by-side appearance via subplots in a row
N_FFT_WM, HOP_WM = 1024, 256
S_orig = np.abs(librosa.stft(orig_audio, n_fft=N_FFT_WM, hop_length=HOP_WM))
S_wm   = np.abs(librosa.stft(wm_audio,   n_fft=N_FFT_WM, hop_length=HOP_WM))
S_diff = np.abs(S_wm - S_orig)

im = axes[1].imshow(
    librosa.amplitude_to_db(S_diff, ref=np.max),
    origin="lower", aspect="auto", cmap="magma",
    extent=[0, len(orig_audio)/sr, 0, sr/2],
)
axes[1].axhline(WM_FREQ_HZ, color="cyan", linestyle="--", lw=1.5,
                label=f"Watermark frequency = {WM_FREQ_HZ} Hz")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Frequency (Hz)")
axes[1].set_title("Spectrogram of the *difference* (watermarked − original) — the watermark is the bright line")
axes[1].legend(loc="upper right")
fig.colorbar(im, ax=axes[1], format="%+0.1f dB")

# 3) Frequency slice — magnitude at each frequency, averaged over time
freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT_WM)
mag_orig = S_orig.mean(axis=1)
mag_wm   = S_wm.mean(axis=1)

axes[2].plot(freqs, 20*np.log10(mag_orig + 1e-12), color="steelblue", label="Original", lw=1.2)
axes[2].plot(freqs, 20*np.log10(mag_wm   + 1e-12), color="crimson",   label="Watermarked", lw=1.2, alpha=0.8)
axes[2].axvline(WM_FREQ_HZ, color="green", linestyle="--", lw=1.5,
                label=f"Watermark @ {WM_FREQ_HZ} Hz")
axes[2].set_xlabel("Frequency (Hz)")
axes[2].set_ylabel("Magnitude (dB)")
axes[2].set_title("Time-averaged spectrum — see the bump at 4.5 kHz")
axes[2].set_xlim(0, sr/2)
axes[2].legend()

plt.tight_layout()
plt.show()

### 7b — Detecting the watermark automatically

Visual inspection is fine for teaching, useless for production. We need a function that takes a `.wav` and returns a numerical confidence that the watermark is present.

### The detection strategy

We use **matched filtering**, the standard signal-processing technique for detecting a known signal buried in noise. The idea:

1. Generate a **reference watermark** identical to what we'd have embedded (a pure sine at 4500 Hz, same sample rate, same duration as the audio).
2. Compute the **cross-correlation** between the reference and the audio at the watermark frequency band.
3. Normalise: divide the correlation by what we'd expect from random noise at that frequency.

The output is a **detection score**. High score (typically > 5) → watermark present. Low score (< 2) → noise, no watermark.

Equivalently and more directly for a pure-tone watermark: we look at the **energy at exactly 4500 Hz in the spectrum**, normalised against the energy in the surrounding frequency bins. If the bin at 4500 Hz is much louder than its neighbours, the watermark is there.

In [ ]:
# Detector using wider noise bands further away from target

def detect_classical_watermark(
    wav_path:        Path,
    target_freq_hz:  float = WM_FREQ_HZ,
    bandwidth_hz:    float = 30,
    noise_offset_hz: float = 400,    # how far from target the noise bands start
    noise_width_hz:  float = 300,    # how wide each noise band is
) -> dict:
    """Detect a sinusoidal watermark. Uses noise bands offset from the target."""
    audio, sr = sf.read(wav_path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    S = np.abs(librosa.stft(audio, n_fft=8192, hop_length=2048))   # higher freq resolution
    power = (S**2).mean(axis=1)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=8192)

    # Energy in narrow band around watermark frequency
    target_band  = (freqs >= target_freq_hz - bandwidth_hz/2) & \
                   (freqs <= target_freq_hz + bandwidth_hz/2)
    target_energy = power[target_band].mean()

    # Energy in noise bands offset by noise_offset_hz from target
    lower_noise = (freqs >= target_freq_hz - noise_offset_hz - noise_width_hz) & \
                  (freqs <  target_freq_hz - noise_offset_hz)
    upper_noise = (freqs >  target_freq_hz + noise_offset_hz) & \
                  (freqs <= target_freq_hz + noise_offset_hz + noise_width_hz)
    noise_band  = lower_noise | upper_noise
    noise_energy = power[noise_band].mean()

    score = target_energy / (noise_energy + 1e-12)
    DETECTION_THRESHOLD = 10.0

    return {
        "file":           wav_path.name,
        "target_freq":    target_freq_hz,
        "target_energy":  float(target_energy),
        "noise_energy":   float(noise_energy),
        "score":          float(score),
        "is_watermarked": bool(score > DETECTION_THRESHOLD),
        "threshold":      DETECTION_THRESHOLD,
    }


# Re-run the three tests
test_files = [
    DEEPFAKE_DIR / "fake_attack.wav",
    WATERMARK_DIR / "fake_attack_watermarked.wav",
    sorted(AUDIO_DIR.glob("student_*.wav"))[0],
]

print("Classical watermark detection results (improved detector):\n")
detection_log = []
for f in test_files:
    result = detect_classical_watermark(f)
    detection_log.append(result)
    flag = "🔴 WATERMARK DETECTED" if result["is_watermarked"] else "⚪ no watermark"
    print(f"  {f.name:45s}  score={result['score']:>10.2f}   {flag}")

print()
df_detect = pd.DataFrame(detection_log)
df_detect[["file", "score", "is_watermarked", "threshold"]]

### The fragility of classical watermarking — an attacker with two lines of code

A classical sinusoidal watermark survives intact through normal channels — speakers, compression, even moderate background noise. But it's **trivially removable** by anyone who knows what frequency to look for.

The attack:

1. Identify the watermark frequency (in our case: 4500 Hz — but any attacker can find it by visual inspection of the spectrogram, like we did earlier).
2. Apply a **notch filter** — a band-stop filter that zeroes out a narrow frequency band.
3. Save the resulting audio.

The audio still sounds the same. The watermark is gone. The detector returns "no watermark."

This is not a bug in our implementation. It is a **fundamental property of any localised-frequency watermark**: if it's localised, it can be surgically removed. Below we demonstrate the attack on our own watermarked file.

In [ ]:
# The notch filter attack: surgically remove the watermark while keeping audio intelligible
def notch_attack(input_wav: Path, output_wav: Path, freq_hz: float, q: float = 30) -> None:
    """Apply a notch filter at freq_hz to remove a narrowband watermark."""
    audio, sr = sf.read(input_wav)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    # Design notch filter centred at freq_hz with quality factor q (higher q = narrower notch)
    b, a = iirnotch(w0=freq_hz / (sr/2), Q=q)
    attacked = filtfilt(b, a, audio)

    sf.write(output_wav, attacked.astype(np.float32), sr, subtype="PCM_16")


# Apply the attack
ATTACKED_FAKE = WATERMARK_DIR / "fake_attack_watermarked_attacked.wav"
notch_attack(WATERMARKED_FAKE, ATTACKED_FAKE, freq_hz=WM_FREQ_HZ)
print(f"Notch attack applied: removed band at {WM_FREQ_HZ} Hz")
print(f"Saved to {ATTACKED_FAKE.name}\n")

# Listen — should still sound identical
print("Watermarked (with watermark):")
display(Audio(str(WATERMARKED_FAKE)))
print("Watermarked + notch attack (watermark removed):")
display(Audio(str(ATTACKED_FAKE)))

# Re-run the detector on all four files
print("\n" + "=" * 70)
print("Detector results after notch attack:")
print("=" * 70 + "\n")

attack_test_files = [
    DEEPFAKE_DIR / "fake_attack.wav",                                  # synthetic, no wm
    WATERMARKED_FAKE,                                                  # synthetic, wm
    ATTACKED_FAKE,                                                     # synthetic, wm, attacked
    sorted(AUDIO_DIR.glob("student_*.wav"))[0],                        # human, no wm
]

attack_log = []
for f in attack_test_files:
    result = detect_classical_watermark(f)
    attack_log.append(result)
    flag = "🔴 DETECTED" if result["is_watermarked"] else "⚪ no watermark"
    print(f"  {f.name:50s}  score={result['score']:>10.2f}   {flag}")

print()
df_attack = pd.DataFrame(attack_log)
df_attack[["file", "score", "is_watermarked"]]

## 7c — Neural watermarking with AudioSeal

The classical watermark we just built has two fatal weaknesses:

1. **Localised in frequency** — concentrated in one narrow band. A notch filter removes it.
2. **Audio-content-independent** — the watermark is the same sine tone regardless of what's being said. An attacker who knows the frequency can subtract it.

**AudioSeal** (Roman et al., Meta AI, 2024) takes the opposite approach:

- **Distributed across the entire spectrum** — every frequency band carries part of the watermark. A notch filter takes out one slice; the watermark survives in the rest.
- **Audio-content-dependent** — a small neural network generates a unique watermark shaped to the specific audio it's marking. The attacker can't pre-compute a "watermark template" to subtract.
- **Trained for robustness** — the detector was trained against MP3 compression, additive noise, time-stretching, pitch-shifting, and re-recording. It's designed to survive real-world manipulation.
- **Optionally carries a 16-bit message** — beyond "watermark present yes/no", you can encode a 16-bit identifier (e.g. which TTS service generated this audio).

### Architecture (one paragraph, then we use it)

AudioSeal has two neural networks: a **generator** (~38 M parameters) that takes raw audio and produces a watermark tensor of identical shape, and a **detector** (~38 M parameters) that takes any audio and outputs a per-sample probability that the watermark is present. The two were trained jointly with adversarial loss against a suite of audio attacks. We use the pretrained weights — no training needed.

License: MIT. Total download: ~150 MB.

In [ ]:
# Install and load AudioSeal (Meta, 2024)
print("Loading AudioSeal generator (~75 MB)...")
generator = AudioSeal.load_generator("audioseal_wm_16bits")
print("Loading AudioSeal detector (~75 MB)...")
detector  = AudioSeal.load_detector("audioseal_detector_16bits")
print(f"\nAudioSeal ready. Device: cpu")
print(f"Native sample rate: 16 kHz (we'll resample if needed)")

In [ ]:
# Apply AudioSeal watermark and detect it
torch._dynamo.config.suppress_errors = True
torch._dynamo.disable()

NEURAL_WATERMARKED = WATERMARK_DIR / "fake_attack_audioseal.wav"

def apply_audioseal(input_wav: Path, output_wav: Path, message: int = 42) -> dict:
    """Embed an AudioSeal watermark with a 16-bit message."""
    audio, sr = sf.read(input_wav)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != 16_000:
        audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=16_000)
        sr = 16_000

    # AudioSeal expects (batch, channels, samples) float32 tensor
    audio_t = torch.from_numpy(audio.astype(np.float32)).unsqueeze(0).unsqueeze(0)

    # Encode the message as 16-bit binary tensor
    msg_bits = torch.tensor([[int(b) for b in f"{message:016b}"]], dtype=torch.int32)

    # Generate the watermark and add it to the audio
    with torch.no_grad():
        watermark = generator.get_watermark(audio_t, sample_rate=sr, message=msg_bits)
        watermarked = audio_t + watermark

    # Save
    out = watermarked.squeeze().numpy()
    sf.write(output_wav, out, sr, subtype="PCM_16")

    return {
        "input":       input_wav.name,
        "output":      output_wav.name,
        "sample_rate": sr,
        "duration_s":  round(len(out) / sr, 2),
        "message":     message,
        "watermark_rms": round(float(np.sqrt(np.mean(watermark.numpy()**2))), 5),
    }


def detect_audioseal(wav_path: Path) -> dict:
    """Detect AudioSeal watermark + recover the 16-bit message."""
    audio, sr = sf.read(wav_path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != 16_000:
        audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=16_000)
        sr = 16_000

    audio_t = torch.from_numpy(audio.astype(np.float32)).unsqueeze(0).unsqueeze(0)

    with torch.no_grad():
        # `detect_watermark` returns (probability per sample, decoded message bits)
        result, message_bits = detector.detect_watermark(audio_t, sample_rate=sr)

    confidence = float(result)
    msg_bits   = message_bits.squeeze().tolist()
    msg_int    = int("".join(str(int(b)) for b in msg_bits), 2)

    return {
        "file":           wav_path.name,
        "confidence":     round(confidence, 4),
        "is_watermarked": confidence > 0.5,
        "decoded_msg":    msg_int,
        "decoded_bits":   "".join(str(int(b)) for b in msg_bits),
    }


# 1) Apply the neural watermark
meta = apply_audioseal(DEEPFAKE_DIR / "fake_attack.wav", NEURAL_WATERMARKED, message=42)
print("AudioSeal watermark applied:")
for k, v in meta.items():
    print(f"  {k:15s}: {v}")
print()

# 2) Quick A/B listen
print("Original (no watermark):")
display(Audio(str(DEEPFAKE_DIR / "fake_attack.wav")))
print("AudioSeal-watermarked:")
display(Audio(str(NEURAL_WATERMARKED)))

# 3) Detect on all four files
print("\n" + "=" * 70)
print("AudioSeal detection results:")
print("=" * 70 + "\n")

audioseal_test_files = [
    DEEPFAKE_DIR  / "fake_attack.wav",                          # synthetic, no wm
    NEURAL_WATERMARKED,                                         # synthetic, AudioSeal wm
    sorted(AUDIO_DIR.glob("student_*.wav"))[0],                 # human, no wm
]

audioseal_log = []
for f in audioseal_test_files:
    result = detect_audioseal(f)
    audioseal_log.append(result)
    flag = "🔴 DETECTED" if result["is_watermarked"] else "⚪ no watermark"
    msg_str = f"   message={result['decoded_msg']:>5d}" if result["is_watermarked"] else ""
    print(f"  {f.name:45s}  conf={result['confidence']:.4f}   {flag}{msg_str}")

print()
df_audioseal = pd.DataFrame(audioseal_log)
df_audioseal

### The same attack, on the neural watermark

Earlier we destroyed the classical watermark with a notch filter at 4500 Hz — score collapsed from 71 to 0. Now we apply **exactly the same attack** to the AudioSeal-watermarked file and see what happens.

The expectation: AudioSeal survives. The watermark is distributed across the whole spectrum, so notching out 4500 Hz removes only a tiny fraction of it. The detector should still find it with high confidence.

We also try a more aggressive attack: **MP3 compression at low bitrate** — the kind of degradation that happens when audio is uploaded to social media or messaging apps. This is what AudioSeal was specifically trained to survive.

In [ ]:
# Attack the AudioSeal watermark with the same techniques that destroyed the classical one

# 1) Notch attack
NOTCHED_NEURAL = WATERMARK_DIR / "fake_attack_audioseal_notched.wav"
notch_attack(NEURAL_WATERMARKED, NOTCHED_NEURAL, freq_hz=WM_FREQ_HZ)
print(f"Notch attack at {WM_FREQ_HZ} Hz applied to AudioSeal-watermarked file.\n")

# 2) Heavy additive noise attack (simulates poor recording conditions)
NOISY_NEURAL = WATERMARK_DIR / "fake_attack_audioseal_noisy.wav"
audio, sr = sf.read(NEURAL_WATERMARKED)
noise = np.random.normal(0, 0.02, len(audio)).astype(np.float32)   # 2% white noise
sf.write(NOISY_NEURAL, audio + noise, sr, subtype="PCM_16")
print(f"Additive Gaussian noise (sigma=0.02) applied to AudioSeal-watermarked file.\n")

# 3) Aggressive volume reduction (simulates quiet capture)
QUIET_NEURAL = WATERMARK_DIR / "fake_attack_audioseal_quiet.wav"
audio, sr = sf.read(NEURAL_WATERMARKED)
sf.write(QUIET_NEURAL, audio * 0.1, sr, subtype="PCM_16")
print(f"Volume reduced to 10% (simulates distant or muffled capture).\n")


# Run AudioSeal detector on all attacked variants
print("=" * 70)
print("AudioSeal detection under attack:")
print("=" * 70 + "\n")

attacked_files = [
    NEURAL_WATERMARKED,        # baseline: clean watermarked
    NOTCHED_NEURAL,            # notch attack
    NOISY_NEURAL,              # noise attack
    QUIET_NEURAL,              # volume attack
]

robustness_log = []
for f in attacked_files:
    result = detect_audioseal(f)
    robustness_log.append(result)
    flag = "🔴 DETECTED" if result["is_watermarked"] else "⚪ no watermark"
    msg = f"   msg={result['decoded_msg']}" if result["is_watermarked"] else ""
    print(f"  {f.name:50s}  conf={result['confidence']:.4f}  {flag}{msg}")

print()
df_robust = pd.DataFrame(robustness_log)
df_robust

### Visualising the AudioSeal watermark

The classical watermark was visible as a bright horizontal line at exactly 4500 Hz in the spectrogram. AudioSeal works differently — the watermark is **distributed across the entire frequency range and shaped by the audio content itself**. So we expect a *very different* visual signature:

1. **In the time domain**: the watermark is a tiny perturbation, varying in shape — not a clean sinusoid.
2. **In the spectrogram**: the watermark spreads across all frequencies. No single line to point at.
3. **In the time-averaged spectrum**: differences are *everywhere*, not concentrated at one frequency.

This is exactly why the notch attack failed against AudioSeal — there is no single frequency to notch out.

In [ ]:
# Visualise the AudioSeal watermark vs the classical one — side-by-side comparison

orig_audio,    sr = sf.read(DEEPFAKE_DIR / "fake_attack.wav")
classical_wm,  _  = sf.read(WATERMARKED_FAKE)
neural_audio,  sr_n = sf.read(NEURAL_WATERMARKED)

# AudioSeal works at 16 kHz; the original is 22050 Hz. Resample original to 16 kHz for fair comparison.
if sr != sr_n:
    orig_audio_16k = librosa.resample(orig_audio.astype(np.float32), orig_sr=sr, target_sr=sr_n)
else:
    orig_audio_16k = orig_audio

# Align lengths
min_len = min(len(orig_audio_16k), len(neural_audio))
orig_audio_16k = orig_audio_16k[:min_len]
neural_audio   = neural_audio[:min_len]
neural_wm_only = neural_audio - orig_audio_16k   # the AudioSeal watermark, isolated

fig, axes = plt.subplots(3, 2, figsize=(16, 12))

# ============ Row 1: Time-domain watermark (×20 for visibility) ============
t_classical = np.arange(len(classical_wm)) / sr
classical_wm_only = classical_wm[:len(t_classical)] - orig_audio[:len(t_classical)]

axes[0, 0].plot(t_classical, classical_wm_only * 20, color="crimson", lw=0.4)
axes[0, 0].set_title("Classical watermark (time, ×20)")
axes[0, 0].set_xlabel("Time (s)")
axes[0, 0].set_ylabel("Amplitude")
axes[0, 0].set_ylim(-0.5, 0.5)

t_neural = np.arange(len(neural_wm_only)) / sr_n
axes[0, 1].plot(t_neural, neural_wm_only * 20, color="darkviolet", lw=0.4)
axes[0, 1].set_title("AudioSeal watermark (time, ×20)")
axes[0, 1].set_xlabel("Time (s)")
axes[0, 1].set_ylabel("Amplitude")
axes[0, 1].set_ylim(-0.5, 0.5)

# ============ Row 2: Spectrogram of the watermark only ============
N_FFT_WM, HOP_WM = 1024, 256

S_classical = np.abs(librosa.stft(classical_wm_only.astype(np.float32),
                                  n_fft=N_FFT_WM, hop_length=HOP_WM))
im0 = axes[1, 0].imshow(librosa.amplitude_to_db(S_classical, ref=np.max),
                        origin="lower", aspect="auto", cmap="magma",
                        extent=[0, len(classical_wm_only)/sr, 0, sr/2])
axes[1, 0].axhline(WM_FREQ_HZ, color="cyan", linestyle="--", lw=1.2)
axes[1, 0].set_title("Classical watermark spectrogram — single bright line @ 4.5 kHz")
axes[1, 0].set_ylabel("Frequency (Hz)")
axes[1, 0].set_xlabel("Time (s)")
fig.colorbar(im0, ax=axes[1, 0], format="%+0.0f dB")

S_neural = np.abs(librosa.stft(neural_wm_only.astype(np.float32),
                               n_fft=N_FFT_WM, hop_length=HOP_WM))
im1 = axes[1, 1].imshow(librosa.amplitude_to_db(S_neural, ref=np.max),
                        origin="lower", aspect="auto", cmap="magma",
                        extent=[0, len(neural_wm_only)/sr_n, 0, sr_n/2])
axes[1, 1].set_title("AudioSeal watermark spectrogram — energy spread across ALL frequencies")
axes[1, 1].set_ylabel("Frequency (Hz)")
axes[1, 1].set_xlabel("Time (s)")
fig.colorbar(im1, ax=axes[1, 1], format="%+0.0f dB")

# ============ Row 3: Time-averaged spectrum (original vs watermarked) ============
freqs_classical = librosa.fft_frequencies(sr=sr,  n_fft=N_FFT_WM)
S_orig_c    = np.abs(librosa.stft(orig_audio.astype(np.float32),  n_fft=N_FFT_WM, hop_length=HOP_WM)).mean(axis=1)
S_wm_c      = np.abs(librosa.stft(classical_wm.astype(np.float32), n_fft=N_FFT_WM, hop_length=HOP_WM)).mean(axis=1)

axes[2, 0].plot(freqs_classical, 20*np.log10(S_orig_c + 1e-12), color="steelblue", lw=1.2, label="Original")
axes[2, 0].plot(freqs_classical, 20*np.log10(S_wm_c   + 1e-12), color="crimson",   lw=1.2, alpha=0.8, label="Watermarked")
axes[2, 0].axvline(WM_FREQ_HZ, color="green", linestyle="--", lw=1.2, label=f"@ {WM_FREQ_HZ} Hz")
axes[2, 0].set_title("Classical — narrow spike at 4.5 kHz, everything else identical")
axes[2, 0].set_xlabel("Frequency (Hz)")
axes[2, 0].set_ylabel("Magnitude (dB)")
axes[2, 0].set_xlim(0, sr/2)
axes[2, 0].legend()

freqs_neural = librosa.fft_frequencies(sr=sr_n, n_fft=N_FFT_WM)
S_orig_n = np.abs(librosa.stft(orig_audio_16k.astype(np.float32),  n_fft=N_FFT_WM, hop_length=HOP_WM)).mean(axis=1)
S_wm_n   = np.abs(librosa.stft(neural_audio.astype(np.float32),    n_fft=N_FFT_WM, hop_length=HOP_WM)).mean(axis=1)

axes[2, 1].plot(freqs_neural, 20*np.log10(S_orig_n + 1e-12), color="steelblue", lw=1.2, label="Original")
axes[2, 1].plot(freqs_neural, 20*np.log10(S_wm_n   + 1e-12), color="darkviolet", lw=1.2, alpha=0.8, label="AudioSeal-watermarked")
axes[2, 1].set_title("AudioSeal — small differences EVERYWHERE, no single spike to attack")
axes[2, 1].set_xlabel("Frequency (Hz)")
axes[2, 1].set_ylabel("Magnitude (dB)")
axes[2, 1].set_xlim(0, sr_n/2)
axes[2, 1].legend()

plt.tight_layout()
plt.show()

### What you're seeing in the six panels

**Row 1 — the watermark as a time-domain signal (amplified ×20 for visibility):**
- *Classical*: a perfectly regular sinusoid. Every cycle is identical. The signal is predictable and content-independent.
- *AudioSeal*: an irregular signal that **follows the shape of the audio** — energy concentrated where the voice is, near-zero during silence. The watermark is content-aware.

**Row 2 — spectrogram of the watermark alone (after subtracting the original audio):**
- *Classical*: a single bright horizontal line at 4500 Hz, everything else dark. **One target for an attacker.**
- *AudioSeal*: energy spread across the entire time-frequency plane. **No single frequency to remove.**

**Row 3 — time-averaged spectrum, original vs watermarked, overlaid:**
- *Classical*: two curves that overlap perfectly except for an isolated spike at 4500 Hz.
- *AudioSeal*: two curves that diverge slightly **everywhere across the spectrum**. The watermark hides in plain sight, distributed across every frequency bin.

This is why the notch attack destroyed the classical watermark but left AudioSeal at confidence 1.0. **The classical watermark is a target. The AudioSeal watermark is the medium itself.**

## What have we learned?

Let's discuss in class.


### Summary of the workshop

Across this notebook we built — from scratch — a complete voice biometrics and deepfake-defence pipeline. Here's what's on disk and in your head by the end:

#### Part 1 — From raw audio to speaker identity (refresher)
We loaded LibriSpeech, extracted MFCCs as a pedagogical baseline, and ran every clip through **WavLM-Base-Plus-SV** to get 512-dim L2-normalised speaker embeddings. We saw that cosine similarity between these embeddings clusters by speaker identity, *not by what was said*.

#### Part 2 — Enrolling your own voice
You recorded three clips and joined the bank as the 11th identity. We measured your **intra-recording consistency** and confirmed your enrollment was clean (cosine > 0.95 across your own clips).

#### Part 3 — Persisting voices to a database
We built a SQLite biometrics database with one row per identity. Each row stores a single **centroid** — the average of three L2-normalised utterance embeddings, re-normalised. Raw audio gets discarded; only the biometric template lives in the DB. This mirrors how real production systems handle voice data for GDPR / data-minimisation reasons.

#### Part 4 — Calibrating a verification threshold
We computed every possible pair of utterances (genuine + impostor), built the **two score distributions**, and located two production-grade operating points:
- **EER** — Equal Error Rate, the "neutral" cut where false-accept and false-reject rates are equal.
- **FAR=1%** — the conservative cut where only 1% of impostor pairs are wrongly accepted. We used this one.

We learned that **threshold selection is a policy decision**, not a math one: it depends on whether your application punishes false accepts or false rejects more harshly.

#### Part 5 — The verify() function
We turned the database + threshold into a one-line `verify(wav_path)` call that returns the top match and the top-3 candidates with their scores. The smoke test (each enrollment utterance against the full DB) hit 100% accuracy.

#### Part 6 — The battle demo and the lesson on session drift
You attempted to verify a fresh, unseen recording against your own enrollment — possibly with surprising results. You may have been misidentified as another speaker, or matched yourself by a thinner margin than expected.

**The lesson**: WavLM embeddings encode "how a voice sounds in a recording", not "who a person biologically is." Variation in microphone, room, mood, and especially **linguistic content** moves the embedding in 512-D space. This is why every production voice biometrics system uses a **text-dependent passphrase** during both enrollment and verification: to keep the embedding distance stable across sessions.

#### Part 7 — Synthetic audio and watermarking
We generated TTS audio with Microsoft SAPI5 and ran it against our biometric database. All three deepfakes were correctly rejected — but only because the SAPI5 acoustic profile is *coincidentally* far from any real voice in our bank. A modern voice-cloning TTS (XTTS, ElevenLabs) would land near a real centroid and pass the threshold. **The verifier does not detect "fakeness". It detects "distance to nearest enrolled human".**

So we built a separate defence: **audio watermarking**.

- **Classical watermark** — a single sinusoid at 4500 Hz, easy to embed, easy to detect, **destroyed by a 4-line notch filter**.
- **AudioSeal (Meta, 2024)** — a neural watermark distributed across the entire spectrum and shaped by the audio content. It survived the notch attack and the volume attack at confidence 1.0. It carried a recoverable 16-bit message (decoded perfectly: 42). It only broke under heavy white-noise injection — and even there, the design philosophy is honest: AudioSeal raises the cost of an attack by orders of magnitude, but no watermark is invincible against an adversary willing to destroy audio quality.

---

### The honest closing message

Voice biometrics in 2026 is not magic. It's a similarity threshold over a deep-learning embedding space. It is brittle to session drift, defeatable by voice cloning, and never deployed alone in production — always paired with passphrases, device fingerprints, and multi-factor authentication.

Watermarking is the modern complement to verification: instead of asking "whose voice does this sound like?", it asks "was this audio generated by a system that signed its output?". This is the direction the industry is moving (OpenAI, Meta, Google all publish watermarking research as of 2025), but it requires **cooperation from the TTS providers**. A watermark only helps if the generator embeds it.

The arms race between voice synthesis and voice authentication is not going to be won by one side. Your job, as engineers, is to know **what each tool actually does, what it doesn't, and what compensating controls you need to layer around it**.

That's the workshop. :)  
<br> <br> <br>

I want to personally thank you for your time and interest in this amazing -and yet short- learning journey; hope you found it useful!

Fede.
